# Jitter
The aim of this project is to develop a fast sentiment analysis of headlines.  The system is focused on economic jitterness, but can be easily refocused to other topics.

### Architecture
The data pipeline is as follows:

1. Obtain headlines from a list of RSS feeds.
2. Embed headlines (via FinBERT).
3. Score each headline by cosine similarity with preset extreme uncertainty sentences.
4. Average daily scores.

### Demonstration
#### Config
We begin by loading the model. Because we focus on economic sentiment, we will use [FinBERT](https://huggingface.co/ProsusAI/finbert).

In [1]:
from embedding import HFEmbedder

model = "ProsusAI/finbert"
embed = HFEmbedder(model)

dim = embed.dimension()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Next we load the preset centroid sentences that will provide the anchor for sentiment analysis.

In [2]:
import pandas as pd

centroid_sentences = pd.read_csv("centroid.csv")

We compute the embedding for each sentence.

In [3]:
centroid_sentences["embedding"] = centroid_sentences["sentence"].apply(embed)

We compute the centroid vector.

In [4]:
centroid = sum(centroid_sentences.embedding) / len(centroid_sentences.embedding)

**Note.** We could instead compute category-wise centroid vectors.  This would allow scoring for each specific source of jitter.

#### Data retrieval and processing
We download the feed from our feeds list.

In [5]:
from sourcing import from_urls

sources = pd.read_csv("sources.csv")
feed = from_urls(sources.url)

Remove the duplicates.

In [6]:
feed.drop_duplicates(subset="title", inplace=True)

In [7]:
feed

,title,description,timestamp,source
0,Why Keir Starmer Remains in Deep Peril After S...,Prime Minister Keir Starmer seemed to have won...,"(2026, 5, 12, 19, 55, 4, 1, 132, 0)",NYT > World News
1,Xi Is Poised to Press Trump on Arms Sales to T...,Beijing has called Taiwan the “core of China’s...,"(2026, 5, 12, 6, 21, 56, 1, 132, 0)",NYT > World News
2,Iran War Live Updates: Trump Says Iran Must Ma...,Iran stuck to its most recent demands after Pr...,"(2026, 5, 12, 19, 56, 41, 1, 132, 0)",NYT > World News
3,Russia Keeps Attacking U.S. Firms in Ukraine. ...,"Facilities tied to Coca-Cola, Cargill, Mondele...","(2026, 5, 12, 10, 46, 19, 1, 132, 0)",NYT > World News
4,Putin’s Credibility Has Suffered in the Iran W...,Despite its long slog in Ukraine and the loss ...,"(2026, 5, 12, 14, 22, 33, 1, 132, 0)",NYT > World News
...,...,...,...,...
398,Elon Musk’s Confidante Shivon Zilis Is Cast as...,Shivon Zilis worked closely with Elon Musk whi...,"(2026, 5, 6, 23, 50, 25, 2, 126, 0)",NYT > Technology
399,Meditating or Rebooting? A Robot Buddhist Monk...,"Gabi, the newest monk at a temple in Seoul tha...","(2026, 5, 7, 9, 24, 59, 3, 127, 0)",NYT > Technology
400,American Factories Lag in Adopting A.I. This D...,A Bristol Myers Squibb plant that makes cancer...,"(2026, 5, 12, 4, 17, 14, 1, 132, 0)",NYT > Technology
401,"Elon Musk Wanted OpenAI to Go Commercial, Greg...","Greg Brockman, OpenAI’s president, testified i...","(2026, 5, 6, 0, 52, 4, 2, 126, 0)",NYT > Technology


Next, we compute the embedding of each entry.  To do this, we pass `"{title} | {description}"` to our embedding function.

In [8]:
feed["embedding"] = feed.apply(
    lambda x: embed(x["title"] + " | " + x["description"], method="CLS"), axis=1
)

Then we score each embedding using cosine-similarity with respect to the centroid tensor.

In [9]:
from torch.nn.functional import cosine_similarity

feed["score"] = feed["embedding"].apply(
    lambda x: float(cosine_similarity(x, centroid, dim=1))
)

In [10]:
feed

,title,description,timestamp,source,embedding,score
0,Why Keir Starmer Remains in Deep Peril After S...,Prime Minister Keir Starmer seemed to have won...,"(2026, 5, 12, 19, 55, 4, 1, 132, 0)",NYT > World News,"[[tensor(-0.0215), tensor(-0.0631), tensor(0.2...",0.435799
1,Xi Is Poised to Press Trump on Arms Sales to T...,Beijing has called Taiwan the “core of China’s...,"(2026, 5, 12, 6, 21, 56, 1, 132, 0)",NYT > World News,"[[tensor(-0.1535), tensor(0.1109), tensor(-0.7...",0.251794
2,Iran War Live Updates: Trump Says Iran Must Ma...,Iran stuck to its most recent demands after Pr...,"(2026, 5, 12, 19, 56, 41, 1, 132, 0)",NYT > World News,"[[tensor(0.0086), tensor(-0.0458), tensor(0.07...",0.456987
3,Russia Keeps Attacking U.S. Firms in Ukraine. ...,"Facilities tied to Coca-Cola, Cargill, Mondele...","(2026, 5, 12, 10, 46, 19, 1, 132, 0)",NYT > World News,"[[tensor(-0.2028), tensor(-0.5459), tensor(-0....",0.438467
4,Putin’s Credibility Has Suffered in the Iran W...,Despite its long slog in Ukraine and the loss ...,"(2026, 5, 12, 14, 22, 33, 1, 132, 0)",NYT > World News,"[[tensor(0.3844), tensor(0.3633), tensor(-0.54...",0.210972
...,...,...,...,...,...,...
398,Elon Musk’s Confidante Shivon Zilis Is Cast as...,Shivon Zilis worked closely with Elon Musk whi...,"(2026, 5, 6, 23, 50, 25, 2, 126, 0)",NYT > Technology,"[[tensor(-0.1985), tensor(0.3876), tensor(-0.6...",0.315318
399,Meditating or Rebooting? A Robot Buddhist Monk...,"Gabi, the newest monk at a temple in Seoul tha...","(2026, 5, 7, 9, 24, 59, 3, 127, 0)",NYT > Technology,"[[tensor(0.3870), tensor(0.4047), tensor(-0.23...",0.295478
400,American Factories Lag in Adopting A.I. This D...,A Bristol Myers Squibb plant that makes cancer...,"(2026, 5, 12, 4, 17, 14, 1, 132, 0)",NYT > Technology,"[[tensor(0.3333), tensor(0.1633), tensor(-0.74...",0.064972
401,"Elon Musk Wanted OpenAI to Go Commercial, Greg...","Greg Brockman, OpenAI’s president, testified i...","(2026, 5, 6, 0, 52, 4, 2, 126, 0)",NYT > Technology,"[[tensor(-0.2337), tensor(0.7252), tensor(-0.8...",0.265689


In [11]:
feed.score.median()

np.float64(0.3067932724952698)

# Improvements
The current method does not discriminate between relevant news items and irrelevant ones (e.g. soccer news).  There are of course several directions of improvement, one of them being to first filter items by relevance and then compute scores for the remaining items, possibly weighting by relevance again.

As a first approximation, we use `facebook/bart-large-mnli` for zero-shot classification.

In [12]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
relevant = ["politics", "geopolitics", "economy", "finance"]
irrelevant = ["sport", "art", "culture"]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [13]:
sentence = "Come see this new exhibit at the MET"
classifier(sentence, relevant + irrelevant, multi_label=True)

{'sequence': 'Come see this new exhibit at the MET',
 'labels': ['art',
  'culture',
  'economy',
  'geopolitics',
  'finance',
  'politics',
  'sport'],
 'scores': [0.9456347227096558,
  0.7331657409667969,
  0.22936363518238068,
  0.18789194524288177,
  0.160027414560318,
  0.08392024785280228,
  0.05734916031360626]}

In [14]:
classifier(sentence, ["jittery"])

{'sequence': 'Come see this new exhibit at the MET',
 'labels': ['jittery'],
 'scores': [0.14531594514846802]}

In [15]:
type(embed(sentence))

torch.Tensor

This seems to be working decently well for both filtering on relevance and for evaluating "jittery".  However, the model is fairly large and slow, and thus could be a good teacher for a simple neural network student model.